## N그램과 BLEU(N-Gram and Bilingual Evaluation Understudy)

N그램 기반 정확도 계산

$$ P(w_i \mid w_{i-n+1}, \dots, w_{i-1}) \approx \frac{C(w_{i-n+1}, \dots, w_i)}{C(w_{i-n+1}, \dots, w_{i-1})} $$

여기서 각 기호의 의미는 다음과 같음.
* $w_i$: 현재 예측하려는 $i$번째 단어
* $w_{i-n+1}, \dots, w_{i-1}$: 현재 단어 앞에 나온 $n-1$개의 문맥(Context) 단어들
* $C(\dots)$: 해당 단어 조합이 전체 말뭉치(Corpus)에서 등장한 횟수(Count)
* $n$: 한 번에 묶어서 볼 단어의 개수 (Unigram($n=1$), Bigram($n=2$), Trigram($n=3$))

예를 들어, "I love deep learning" 문장에서 "love deep"으로 다음 단어 예측이라면, 
$$ P(\text{learning} \mid \text{love, deep}) \approx \frac{C(\text{love, deep, learning})}{C(\text{love, deep})} = \frac{80}{100} = 0.8 $$

* **$w_i$**: "learning" (예측하려는 현재 단어)
* **$w_{i-2}, w_{i-1}$**: "love", "deep" (앞에 나온 $n-1$개의 힌트 단어)
* **$C(\text{love, deep})$**: 100회 (말뭉치에서 "love deep"이 등장한 총 횟수)
* **$C(\text{love, deep, learning})$**: 80회 ("love deep" 뒤에 "learning"이 이어서 나온 횟수)
* **$n=3$**: 한 번에 3개의 단어 덩어리를 고려함 (Trigram)

In [1]:
from collections import Counter
import numpy as np

# N-Gram 추출 함수
def get_ngrams(text, n):
    words = text.split()
    return [tuple(words[i:i+n]) for i in range(len(words)-n+1)]

text = "The quick brown fox jumps over the lazy dog" # 예시 텍스트

print("N-Gram 예시 ")
for n in range(1, 4):
    ngrams = get_ngrams(text, n)
    print(f"{n}-gram: {ngrams}")

# N-gram 빈도 분석
print(f"\nBigram 빈도 분석 ")
bigrams = get_ngrams(text, 2)
bigram_counts = Counter(bigrams)
for bigram, count in bigram_counts.items():
    print(f"'{' '.join(bigram)}': {count}")


N-Gram 예시 
1-gram: [('The',), ('quick',), ('brown',), ('fox',), ('jumps',), ('over',), ('the',), ('lazy',), ('dog',)]
2-gram: [('The', 'quick'), ('quick', 'brown'), ('brown', 'fox'), ('fox', 'jumps'), ('jumps', 'over'), ('over', 'the'), ('the', 'lazy'), ('lazy', 'dog')]
3-gram: [('The', 'quick', 'brown'), ('quick', 'brown', 'fox'), ('brown', 'fox', 'jumps'), ('fox', 'jumps', 'over'), ('jumps', 'over', 'the'), ('over', 'the', 'lazy'), ('the', 'lazy', 'dog')]

Bigram 빈도 분석 
'The quick': 1
'quick brown': 1
'brown fox': 1
'fox jumps': 1
'jumps over': 1
'over the': 1
'the lazy': 1
'lazy dog': 1


In [2]:
texts = [
    "I love machine learning",
    "Machine learning is amazing", 
    "Deep learning loves data"
] # 다양한 텍스트로 테스트

print(f"\n 여러 텍스트의 Bigram ")
for text in texts:
    bigrams = get_ngrams(text, 2)
    print(f"'{text}' → {bigrams}")


 여러 텍스트의 Bigram 
'I love machine learning' → [('I', 'love'), ('love', 'machine'), ('machine', 'learning')]
'Machine learning is amazing' → [('Machine', 'learning'), ('learning', 'is'), ('is', 'amazing')]
'Deep learning loves data' → [('Deep', 'learning'), ('learning', 'loves'), ('loves', 'data')]


BLEU 계산

$$ BP = \begin{cases} 1 & \text{if } c > r \\ e^{(1-r/c)} & \text{if } c \leq r \end{cases} $$
* $c$: 모델이 만든 문장의 길이 (Candidate length)
* $r$: 정답 문장의 길이 (Reference length)

후보 문장이 정답과 같으면 ($c=r$): 지수가 $1-1=0$이 됨. $e^0 = \mathbf{1}$이 되어 패널티 없음.
후보 문장이 짧아질수록 $e$의 지수가 점점 더 큰 음수가 되어, 전체 값이 0에 가까워짐.

In [3]:
def simple_bleu(reference, candidate, n=2): # BLEU Score 계산
    ref_ngrams = Counter(get_ngrams(reference, n))
    cand_ngrams = Counter(get_ngrams(candidate, n))
    
    # n-gram 매칭 개수
    matches = sum((ref_ngrams & cand_ngrams).values())
    total = sum(cand_ngrams.values())
    
    return matches / total if total > 0 else 0

print("BLEU Score 예시")
reference = "The cat is on the mat"  # 정답 문장
candidates = [
    "The cat is on the mat",        # 완전 일치
    "A cat is on the mat",          # 거의 일치  
    "The dog is on the bed",        # 부분 일치
    "Hello world programming"       # 완전 다름
]

for candidate in candidates:
    bleu = simple_bleu(reference, candidate, n=2)
    print(f"정답: '{reference}'")
    print(f"후보: '{candidate}'")
    print(f"BLEU: {bleu:.3f}")
    print()


BLEU Score 예시
정답: 'The cat is on the mat'
후보: 'The cat is on the mat'
BLEU: 1.000

정답: 'The cat is on the mat'
후보: 'A cat is on the mat'
BLEU: 0.800

정답: 'The cat is on the mat'
후보: 'The dog is on the bed'
BLEU: 0.400

정답: 'The cat is on the mat'
후보: 'Hello world programming'
BLEU: 0.000



In [ ]:
print("번역 품질 평가 예시")
ref_ko = "고양이가 매트 위에 있다"
translations = [
    "고양이가 매트 위에 있다",      # 완벽한 번역
    "고양이는 매트 위에 있다",      # 조사 다름
    "고양이가 침대 위에 있다",      # 단어 다름
    "개가 매트 위에 있다"          # 주체 다름
]

for trans in translations:
    bleu = simple_bleu(ref_ko, trans, n=1)  # 한글은 1-gram으로
    print(f"번역: '{trans}' → BLEU: {bleu:.3f}")

번역 품질 평가 예시
번역: '고양이가 매트 위에 있다' → BLEU: 1.000
번역: '고양이는 매트 위에 있다' → BLEU: 0.750
번역: '고양이가 침대 위에 있다' → BLEU: 0.750
번역: '개가 매트 위에 있다' → BLEU: 0.750
